In [ ]:
#01 Install dependencies & mount Drive
!pip -q install opencv-python pandas tqdm pillow

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
#02 Locate and extract the dataset
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/split_dataset.zip")
EXTRACT_ROOT = Path("/content/split_dataset")

assert ZIP_PATH.exists(), f"Could not find {ZIP_PATH} — check the filename/path on Drive."

already_extracted = EXTRACT_ROOT.exists() and any(EXTRACT_ROOT.rglob("*.jpg"))

if not already_extracted:
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print("Extracting zip (this can take a few minutes for ~230k files)...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_ROOT)
    print("Done extracting.")
else:
    print("Extraction already present — skipping unzip.")

print("EXTRACT_ROOT:", EXTRACT_ROOT)


In [ ]:
from pathlib import Path

INITIAL_EXTRACT_ROOT = Path("/content/split_dataset")
EXTRACT_ROOT = INITIAL_EXTRACT_ROOT # Start with the initial assumption

# Filter out macOS junk (defined early for use in contains_clean_images)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def is_junk(path: Path) -> bool:
    if "__MACOSX" in path.parts:
        return True
    if path.name.startswith("._") or path.name == ".DS_Store":
        return True
    return False

def is_clean_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMG_EXTS and not is_junk(path)

# Function to check if a path contains any clean images
def contains_clean_images(root_path: Path) -> bool:
    if not root_path.is_dir():
        return False
    for p in root_path.rglob("*"):
        if is_clean_image(p):
            return True
    return False

# Try to find a more appropriate EXTRACT_ROOT if initial one yields no images
if EXTRACT_ROOT.exists():
    if not contains_clean_images(EXTRACT_ROOT):
        found_adjusted_root = False
        # Iterate through immediate subdirectories to find a better root
        for item in EXTRACT_ROOT.iterdir():
            if item.is_dir():
                if contains_clean_images(item):
                    original_extract_root = EXTRACT_ROOT
                    EXTRACT_ROOT = item
                    print(f"Adjusted EXTRACT_ROOT from {original_extract_root} to {EXTRACT_ROOT} because it contains image files.")
                    found_adjusted_root = True
                    break # Found a better root, stop searching
        if not found_adjusted_root:
            print(f"Warning: No clean image files found under {INITIAL_EXTRACT_ROOT} or its immediate subdirectories. Check your extraction path or dataset structure.")
else:
    print(f"Error: Initial extraction root {INITIAL_EXTRACT_ROOT} does not exist. Please check extraction process.")

# Now, with the (potentially adjusted) EXTRACT_ROOT, find all images and labels
def is_clean_label(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() == ".txt" and not is_junk(path)

all_images = [p for p in EXTRACT_ROOT.rglob("*") if is_clean_image(p)]
all_labels = [p for p in EXTRACT_ROOT.rglob("*") if is_clean_label(p)]

print(f"Clean images found: {len(all_images):,}")
print(f"Clean label files found: {len(all_labels):,}")

In [ ]:
#04 Build image/label pairs
import pandas as pd
from tqdm import tqdm

def label_path_for(image_path: Path):
    parts = list(image_path.parts)
    if "images" not in parts:
        return None
    idx = parts.index("images")
    label_parts = parts.copy()
    label_parts[idx] = "labels"
    candidate = Path(*label_parts).with_suffix(".txt")
    return candidate if candidate.exists() and not is_junk(candidate) else None

def infer_split_modality(image_path: Path):
    parts = image_path.parts
    idx = parts.index("images")
    modality = parts[idx - 1].upper() if idx - 1 >= 0 else "UNKNOWN"
    split = parts[idx - 2].lower() if idx - 2 >= 0 else "unknown"
    return split, modality

rows = []
skipped_no_images_component = 0
skipped_no_label = 0

for img_path in tqdm(all_images, desc="Pairing images with labels"):
    if "images" not in img_path.parts:
        skipped_no_images_component += 1
        continue

    lbl_path = label_path_for(img_path)
    if lbl_path is None:
        skipped_no_label += 1
        continue

    split, modality = infer_split_modality(img_path)

    rows.append({
        "stem": img_path.stem,
        "split": split,
        "modality": modality,
        "image_path": str(img_path),
        "label_path": str(lbl_path),
    })

manifest_df = pd.DataFrame(rows)

print(f"Paired: {len(manifest_df):,}")
print(f"Skipped (no 'images' folder in path): {skipped_no_images_component:,}")
print(f"Skipped (no matching label found): {skipped_no_label:,}")
print()
print("Pairs by split x modality:")
print(manifest_df.groupby(["split", "modality"]).size().unstack(fill_value=0))


#05 — Sanity checks

Two checks before we trust this manifest:
1. **Class balance** — read every label file's class column (0 = one class, 1 = the
   other, per the dataset's `data.yaml` convention) and tally counts per split/modality.
   This is a full pass since reading small `.txt` files is cheap.
2. **Corrupt-image spot check** — decoding all ~230k images to verify integrity would
   burn a lot of session time for a setup notebook, so we spot-check a random sample
   with `PIL.Image.verify()`. If failures show up, we widen the sample in a rerun.

In [ ]:
from collections import Counter

def read_class_ids(label_path: str):
    text = Path(label_path).read_text().strip()
    if not text:
        return []
    ids = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5:
            ids.append(int(float(parts[0])))
    return ids

class_counter = Counter()
for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Reading class labels"):
    for cid in read_class_ids(row["label_path"]):
        class_counter[(row["split"], row["modality"], cid)] += 1

class_balance_df = pd.DataFrame(
    [{"split": s, "modality": m, "class_id": c, "count": n} for (s, m, c), n in class_counter.items()]
).sort_values(["split", "modality", "class_id"])

print("Class balance (object instances, not images):")
print(class_balance_df.to_string(index=False))


#06 — Save the manifest to Drive

This is the single most important cell in this notebook for the compute-bottleneck
problem: every other notebook (`01`–`07`) loads this CSV instead of re-unzipping and
re-walking the dataset from scratch. Saved to Drive so it survives a runtime
disconnect.

In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/vip_cup_project/manifests")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_DIR / "image_label_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

class_balance_path = OUTPUT_DIR / "class_balance.csv"
class_balance_df.to_csv(class_balance_path, index=False)

print("Saved manifest to:", manifest_path)
print("Saved class balance to:", class_balance_path)


#07 — RGB ↔ IR filename alignment check

The fusion model (`04_model_fusion.ipynb`) stacks RGB and IR into a single 4-channel
input **per frame**, which only works if RGB and IR images are paired by matching
filename stem within the same split. This check tells us up front whether that
assumption holds, before any fusion code gets written against it.

If overlap is low, early fusion by filename won't work as planned and we'd need to
fall back to late fusion, or investigate whether pairing needs to happen by some other
key (timestamp, sequence index) instead of stem.

In [ ]:
alignment_rows = []

for split in sorted(manifest_df["split"].unique()):
    rgb_stems = set(manifest_df[(manifest_df["split"] == split) & (manifest_df["modality"] == "RGB")]["stem"])
    ir_stems = set(manifest_df[(manifest_df["split"] == split) & (manifest_df["modality"] == "IR")]["stem"])

    overlap = rgb_stems & ir_stems
    union = rgb_stems | ir_stems

    alignment_rows.append({
        "split": split,
        "rgb_count": len(rgb_stems),
        "ir_count": len(ir_stems),
        "matched_pairs": len(overlap),
        "overlap_pct_of_union": round(100 * len(overlap) / len(union), 2) if union else 0.0,
    })

alignment_df = pd.DataFrame(alignment_rows)
print(alignment_df.to_string(index=False))

if (alignment_df["overlap_pct_of_union"] < 50).any():
    print()
    print("WARNING: low RGB/IR filename overlap on at least one split.")
    print("Early fusion by filename stem may not be reliable — inspect a few")
    print("unmatched stems below before building 04_model_fusion.ipynb.")


In [ ]:
# Build and save the matched RGB/IR pair manifest that 04_model_fusion.ipynb will consume.
fusion_rows = []

for split in sorted(manifest_df["split"].unique()):
    rgb_df = manifest_df[(manifest_df["split"] == split) & (manifest_df["modality"] == "RGB")].set_index("stem")
    ir_df = manifest_df[(manifest_df["split"] == split) & (manifest_df["modality"] == "IR")].set_index("stem")

    common_stems = rgb_df.index.intersection(ir_df.index)

    for stem in common_stems:
        fusion_rows.append({
            "split": split,
            "stem": stem,
            "rgb_image_path": rgb_df.loc[stem, "image_path"],
            "rgb_label_path": rgb_df.loc[stem, "label_path"],
            "ir_image_path": ir_df.loc[stem, "image_path"],
            "ir_label_path": ir_df.loc[stem, "label_path"],
        })

fusion_pairs_df = pd.DataFrame(fusion_rows)
fusion_manifest_path = OUTPUT_DIR / "fusion_pairs_manifest.csv"
fusion_pairs_df.to_csv(fusion_manifest_path, index=False)

print(f"Matched RGB/IR pairs saved: {len(fusion_pairs_df):,}")
print("Saved to:", fusion_manifest_path)


# 01 — Non-Neural Statistical Baseline (Model 1 proposals + Gaussian classifier)

**Goal:** implement your original alternative-to-a-neural-network idea end to end on
the real VIP Cup dataset — take Model 1's classical proposal boxes (Laplacian-energy
based, no learning involved), crop chips, describe each chip with a handful of
statistics, and classify with a **Gaussian Naive Bayes** model: fit a Gaussian
distribution per feature per class from a batch of real drone/bird chips, and use
Bayes' rule to get `P(drone | features)` directly. This is exactly your original
"take a bunch of images of actual drones and use statistics to obtain our
probability" idea, made concrete.

This notebook reuses your existing Model 1 code (`get_boxes_model1`, the tuned
`Option B` / `Best overlap` parameters, `nms_boxes`, the center-hit matching logic)
rather than re-deriving it — it's been validated in your earlier notebook and this
keeps the comparison to Model 2 (YOLO) fair, since both start from the same manifest.

**A known limitation going in, from your own earlier evaluation:** Model 1's IoU
recall on the real dataset was low (0.02–0.05) even though center-hit recall was
high (0.56–1.0) — meaning its boxes tend to *contain* the object's center without
tightly bounding it. So this notebook matches proposals to ground truth by
**center-hit**, not strict IoU, and that's worth stating plainly in the report as a
constraint of the classical method, not something to hide.

**Class ID mapping — please verify visually.** Your `data.yaml` only has generic
names (`class_0`, `class_1`). Comparing your class-balance numbers (class_0: 28,226
/ class_1: 36,670 total instances) against a published paper using the same official
VIP Cup dataset (which reports ~30,000 bird / ~36,000 drone instances) strongly
suggests **class_id 0 = bird, class_id 1 = drone**. This notebook assumes that
mapping — flag it early if a visual spot-check of a few chips says otherwise, since
it would flip every metric downstream.

**Steps:**
1. Load the manifest from `00_setup_and_data.ipynb`
2. Reuse Model 1's proposal + NMS + matching code
3. Build a labeled chip dataset from a sample of `train` images (statistical
   features + background/bird/drone label)
4. Build the same for a sample of `val` images (evaluation set)
5. Fit Gaussian Naive Bayes on the train chips
6. Evaluate on val chips: accuracy/precision/recall/F1, confusion matrix, and a
   worked example of the actual `P(drone | chip)` output
7. Log the result to the shared model-comparison table on Drive


In [ ]:
#01 — Load the manifest
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path("/content/drive/MyDrive/vip_cup_project/manifests/image_label_manifest.csv")
manifest_df = pd.read_csv(MANIFEST_PATH)

print(f"Loaded manifest: {len(manifest_df):,} image-label pairs")
print(manifest_df.groupby(["split", "modality"]).size().unstack(fill_value=0))


In [ ]:
#02 Model 1: classical proposal method
import cv2
import numpy as np
import torch
import torch.nn.functional as F


def read_image_local(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = img.astype(np.float32)
    if img.max() > 1.5:
        img = img / (65535.0 if img.max() > 255 else 255.0)

    return np.clip(img, 0, 1)


def downsample_binary(mask: np.ndarray, scale: float, mode: str = "any") -> np.ndarray:
    h, w = mask.shape[:2]
    new_h, new_w = max(1, int(h / scale)), max(1, int(w / scale))

    binary01 = (mask > 0).astype(np.float32)
    fraction_on = cv2.resize(binary01, (new_w, new_h), interpolation=cv2.INTER_AREA)

    if mode == "any":
        out = (fraction_on > 0).astype(np.uint8) * 255
    elif mode == "majority":
        out = (fraction_on > 0.5).astype(np.uint8) * 255
    else:
        raise ValueError("mode must be 'any' or 'majority'")

    return out


def downsampled_bbox_to_original(stats, scale):
    x, y, w, h, area = stats
    left = x * scale
    top = y * scale
    right = (x + w) * scale
    bottom = (y + h) * scale
    return left, top, right, bottom


def get_boxes_model1(img, scale=30, max_size=10, energy_threshold=0.01, pool_kernel=11):
    """Model 1: classical proposal-box method. img: RGB or IR image normalized to [0, 1]."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.0)

    lap = cv2.Laplacian(blurred, cv2.CV_32F)
    energy = np.abs(lap)
    energy = energy - np.mean(energy)
    energy = np.tanh(energy)

    binary = (energy > energy_threshold).astype(np.uint8)

    img_t = torch.from_numpy(binary).float()[None, None, :, :]
    pooled = F.max_pool2d(img_t, kernel_size=pool_kernel, stride=1, padding=pool_kernel // 2)
    high_energy = pooled[0, 0].numpy()

    small = downsample_binary(high_energy, scale=scale, mode="majority")
    num_labels, component_labels, stats, centroids = cv2.connectedComponentsWithStats(
        small.astype(np.uint8), connectivity=8
    )

    boxes = []
    for i in range(1, num_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < max_size:
            left, top, right, bottom = downsampled_bbox_to_original(stats[i], scale)
            left = max(0, int(round(left)))
            top = max(0, int(round(top)))
            right = min(img.shape[1], int(round(right)))
            bottom = min(img.shape[0], int(round(bottom)))
            if right > left and bottom > top:
                boxes.append((left, top, right, bottom))

    return boxes


def box_area(box):
    left, top, right, bottom = box
    return max(0, right - left) * max(0, bottom - top)


def compute_iou(box_a, box_b):
    left_a, top_a, right_a, bottom_a = box_a
    left_b, top_b, right_b, bottom_b = box_b

    inter_left, inter_top = max(left_a, left_b), max(top_a, top_b)
    inter_right, inter_bottom = min(right_a, right_b), min(bottom_a, bottom_b)
    inter_area = max(0, inter_right - inter_left) * max(0, inter_bottom - inter_top)

    area_a = max(0, right_a - left_a) * max(0, bottom_a - top_a)
    area_b = max(0, right_b - left_b) * max(0, bottom_b - top_b)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


def nms_boxes(boxes, scores=None, iou_threshold=0.3, top_k=30):
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float) if scores is not None else np.array([box_area(b) for b in boxes])
    order = scores.argsort()[::-1]
    keep = []

    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(keep) >= top_k:
            break
        remaining = order[1:]
        new_remaining = [j for j in remaining if compute_iou(tuple(boxes[i]), tuple(boxes[j])) < iou_threshold]
        order = np.array(new_remaining)

    return [tuple(boxes[i].astype(int)) for i in keep]


def box_center(box):
    left, top, right, bottom = box
    return (left + right) / 2, (top + bottom) / 2


def center_inside_box(center, box):
    cx, cy = center
    left, top, right, bottom = box
    return left <= cx <= right and top <= cy <= bottom


BEST_CHIP_PARAMS = {"scale": 20, "max_size": 100, "energy_threshold": 0.0, "pool_kernel": 11}
NMS_IOU_THRESHOLD = 0.2

print("Model 1 proposal pipeline ready.")


In [ ]:
#03 — YOLO label parsing (reused) + class mapping
def read_yolo_label_file(label_path):
    labels = []
    text = Path(label_path).read_text().strip()
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        labels.append({
            "class_id": int(float(parts[0])),
            "x_center": float(parts[1]),
            "y_center": float(parts[2]),
            "width": float(parts[3]),
            "height": float(parts[4]),
        })
    return labels


def yolo_to_xyxy(label, image_width, image_height):
    x_center = label["x_center"] * image_width
    y_center = label["y_center"] * image_height
    box_width = label["width"] * image_width
    box_height = label["height"] * image_height

    left = max(0, int(round(x_center - box_width / 2)))
    top = max(0, int(round(y_center - box_height / 2)))
    right = min(image_width, int(round(x_center + box_width / 2)))
    bottom = min(image_height, int(round(y_center + box_height / 2)))
    return left, top, right, bottom


# See the note in the intro cell above on how this mapping was inferred — verify visually if in doubt.
CLASS_ID_TO_NAME = {0: "bird", 1: "drone"}


#04 — Build the labeled chip dataset

For each sampled image (RGB only for this baseline, matching Model 1's original
design):
1. Load the image and its ground-truth boxes.
2. Run Model 1 to get candidate proposal boxes, then NMS to remove duplicates.
3. Match each proposal to a ground-truth box by **center-hit** (proposal center
   inside a true box, or true box center inside the proposal — whichever matches).
   A matched proposal inherits that true box's class; an unmatched proposal is
   labeled `background`.
4. Crop the chip and extract statistical features.

Sample sizes are kept modest (`TRAIN_SAMPLE_SIZE`, `VAL_SAMPLE_SIZE`) since this
pipeline runs in pure Python/OpenCV per image rather than a batched GPU op — increase
them later if runtime allows.

In [ ]:
import random

random.seed(42)

TRAIN_SAMPLE_SIZE = 1200
VAL_SAMPLE_SIZE = 400

train_rgb_df = manifest_df[(manifest_df["split"] == "train") & (manifest_df["modality"] == "RGB")]
val_rgb_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == "RGB")]

train_sample = train_rgb_df.sample(n=min(TRAIN_SAMPLE_SIZE, len(train_rgb_df)), random_state=42)
val_sample = val_rgb_df.sample(n=min(VAL_SAMPLE_SIZE, len(val_rgb_df)), random_state=42)

print(f"Train images sampled: {len(train_sample):,}")
print(f"Val images sampled:   {len(val_sample):,}")


In [ ]:
def extract_chip_features(chip):
    """Purely statistical descriptors of a candidate chip — no learned weights."""
    if chip.size == 0 or chip.shape[0] < 2 or chip.shape[1] < 2:
        return None

    gray = cv2.cvtColor((chip * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    lap = cv2.Laplacian(gray, cv2.CV_32F)

    h, w = chip.shape[:2]

    return {
        "width": w,
        "height": h,
        "area": w * h,
        "aspect_ratio": w / h,
        "mean_r": float(chip[:, :, 0].mean()),
        "mean_g": float(chip[:, :, 1].mean()),
        "mean_b": float(chip[:, :, 2].mean()),
        "mean_intensity": float(gray.mean()),
        "std_intensity": float(gray.std()),
        "laplacian_energy_mean": float(np.abs(lap).mean()),
    }


def match_proposal_to_label(proposal_box, true_boxes, true_classes):
    """Center-hit matching (see notebook intro for why, not strict IoU)."""
    prop_center = box_center(proposal_box)

    for true_box, cls in zip(true_boxes, true_classes):
        if center_inside_box(prop_center, true_box):
            return cls
        if center_inside_box(box_center(true_box), proposal_box):
            return cls

    return "background"


FEATURE_COLUMNS = [
    "width", "height", "area", "aspect_ratio",
    "mean_r", "mean_g", "mean_b",
    "mean_intensity", "std_intensity", "laplacian_energy_mean",
]


def build_chip_dataset(image_rows, desc):
    rows = []

    for _, row in tqdm(image_rows.iterrows(), total=len(image_rows), desc=desc):
        img = read_image_local(row["image_path"])
        H, W = img.shape[:2]

        labels = read_yolo_label_file(row["label_path"])
        true_boxes = [yolo_to_xyxy(l, W, H) for l in labels]
        true_classes = [CLASS_ID_TO_NAME.get(l["class_id"], "unknown") for l in labels]

        proposals = get_boxes_model1(img, **BEST_CHIP_PARAMS)
        proposals = nms_boxes(proposals, iou_threshold=NMS_IOU_THRESHOLD)

        for box in proposals:
            left, top, right, bottom = box
            chip = img[top:bottom, left:right]
            features = extract_chip_features(chip)
            if features is None:
                continue

            label = match_proposal_to_label(box, true_boxes, true_classes)
            features["true_label"] = label
            features["image_path"] = row["image_path"]
            rows.append(features)

    return pd.DataFrame(rows)


from tqdm import tqdm

train_chips_df = build_chip_dataset(train_sample, "Building train chips")
val_chips_df = build_chip_dataset(val_sample, "Building val chips")

print()
print("Train chip label counts:")
print(train_chips_df["true_label"].value_counts())
print()
print("Val chip label counts:")
print(val_chips_df["true_label"].value_counts())


#05  Save the chip feature datasets to Drive

Same reasoning as the manifest in `00`: chip generation is the slow part of this
notebook, so cache the result rather than rebuilding it on every rerun.

In [ ]:
FEATURES_DIR = Path("/content/drive/MyDrive/vip_cup_project/features")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

train_chips_df.to_csv(FEATURES_DIR / "baseline_train_chips.csv", index=False)
val_chips_df.to_csv(FEATURES_DIR / "baseline_val_chips.csv", index=False)

print("Saved train/val chip features to:", FEATURES_DIR)


#06 Fit the statistical classifier (Gaussian Naive Bayes)

This is the actual "statistics instead of a neural network" model: Gaussian Naive
Bayes fits a mean and variance per feature, per class, from the training chips, and
classifies new chips using Bayes' rule. Its `predict_proba` output is literally
`P(class | features)` — for `class = "drone"` that's the exact "probability of being
a drone" your original pipeline asked for, with no neural network involved.

We fit on all three classes (`background`, `bird`, `drone`) so the model also learns
to reject Model 1's false-positive proposals, not just discriminate bird vs. drone.

In [ ]:
from sklearn.naive_bayes import GaussianNB

X_train = train_chips_df[FEATURE_COLUMNS].values
y_train = train_chips_df["true_label"].values

clf = GaussianNB()
clf.fit(X_train, y_train)

print("Classes learned:", clf.classes_)


#07 — Evaluate on the val chips

Two views of the results:
1. **Full 3-class report** (background/bird/drone) — diagnostic, shows how well the
   model rejects Model 1's false proposals.
2. **Drone-vs-bird only, on chips whose ground truth is actually an object** — this
   is the number to compare against Model 2 (YOLO)'s classification metrics later,
   since YOLO's accuracy/F1/precision/recall are reported the same way (object
   classes only).

We also print `predict_proba` for a few real val chips as a concrete demonstration
of the probability output.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

X_val = val_chips_df[FEATURE_COLUMNS].values
y_val = val_chips_df["true_label"].values
y_pred = clf.predict(X_val)

print("=== Full 3-class report (background / bird / drone) ===")
print(classification_report(y_val, y_pred, labels=["background", "bird", "drone"], zero_division=0))

print("Confusion matrix (rows = true, cols = predicted; order = background, bird, drone):")
print(confusion_matrix(y_val, y_pred, labels=["background", "bird", "drone"]))


In [ ]:
# Object-only view: comparable to how Model 2 (YOLO) classification metrics will be reported.
object_mask = val_chips_df["true_label"] != "background"
y_val_obj = val_chips_df.loc[object_mask, "true_label"].values
y_pred_obj = y_pred[object_mask.values]

report_dict = classification_report(
    y_val_obj, y_pred_obj,
    labels=["bird", "drone"],
    output_dict=True,
    zero_division=0,
)

accuracy_obj = (y_val_obj == y_pred_obj).mean()
precision_macro = (report_dict["bird"]["precision"] + report_dict["drone"]["precision"]) / 2
recall_macro = (report_dict["bird"]["recall"] + report_dict["drone"]["recall"]) / 2
f1_macro = (report_dict["bird"]["f1-score"] + report_dict["drone"]["f1-score"]) / 2

print("=== Drone-vs-bird only (true object chips) ===")
print(f"Accuracy:  {accuracy_obj:.4f}")
print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro):    {recall_macro:.4f}")
print(f"F1 (macro):        {f1_macro:.4f}")


In [ ]:
# Concrete demonstration of the probability output your original pipeline asked for.
sample_idx = val_chips_df.loc[object_mask].sample(min(5, object_mask.sum()), random_state=1).index
class_order = list(clf.classes_)
drone_idx = class_order.index("drone")

print("Example P(drone | chip) on real val chips with a known ground-truth object:")
for idx in sample_idx:
    row = val_chips_df.loc[idx]
    # Ensure the input to predict_proba is a 2D numpy array of floats
    features_array = np.array(row[FEATURE_COLUMNS].values, dtype=np.float64).reshape(1, -1)
    probs = clf.predict_proba(features_array)[0]
    print(f"  true={row['true_label']:8s}  P(drone)={probs[drone_idx]:.3f}   image={Path(row['image_path']).name}")

#08 — Log this result to the shared model-comparison table

`02_model_rgb.ipynb`, `03_model_ir.ipynb`, and `04_model_fusion.ipynb` will each
append their own row here in the same format, so the final report notebook just
loads one CSV for the full comparison table.

In [ ]:
RESULTS_DIR = Path("/content/drive/MyDrive/vip_cup_project/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "model_comparison.csv"

new_row = {
    "model_name": "Baseline (Model 1 proposals + Gaussian Naive Bayes)",
    "modality": "RGB",
    "accuracy": accuracy_obj,
    "precision": precision_macro,
    "recall": recall_macro,
    "f1": f1_macro,
    "notes": "Statistical, non-neural-network baseline. Object-only (background excluded) metrics.",
}

if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    results_df = results_df[
        ~((results_df["model_name"] == new_row["model_name"]) & (results_df["modality"] == new_row["modality"]))
    ]
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
else:
    results_df = pd.DataFrame([new_row])

results_df.to_csv(RESULTS_PATH, index=False)
print(results_df.to_string(index=False))


# 01 — Model 2 (YOLOv8), RGB-only

**Goal:** train a supervised YOLOv8 detector on the RGB split only, using the clean
manifest from `00_setup_and_data.ipynb`, and evaluate it the way the official VIP
Cup rubric asks for (Accuracy, F1, Precision, Recall).

**What's different from your earlier attempt:**
- Built from the fixed manifest, so no risk of the `0 image-label pairs` bug.
- `imgsz` is set for this dataset's *actual* resolution (~320×256), not carried over
  from a setup tuned for something else — oversized `imgsz` just burns Colab compute
  for no accuracy benefit here.
- Weights and training logs save directly to **Drive**, not `/content`, so a runtime
  disconnect doesn't lose a finished training run.
- A `QUICK_TEST` toggle lets you smoke-test the whole pipeline on a small subset
  before committing a long run to the full ~40k training images.

**On "Accuracy/F1/Precision/Recall"**: Ultralytics reports Precision, Recall, and F1
per class directly (matched at IoU≥0.5) — those map cleanly onto the rubric. There is
no single standard "accuracy" scalar in object detection the way there is in
classification, so this notebook reports **mAP@0.5** as the accuracy figure, with
that substitution stated explicitly rather than silently. If your VIP Cup rubric
provides an official scoring script with a different accuracy definition, swap it in
before final submission — flag this in the report.

**Steps:**
1. Load the manifest, filter to RGB
2. Build a YOLO-format dataset directory (via symlinks — fast, no duplicate storage)
3. Train YOLOv8n
4. Evaluate on val (and test) with Ultralytics' built-in metrics
5. Log results to the shared `model_comparison.csv`
6. Save the weights path for later notebooks (`05` robustness, `06` speed, `07` demo)


In [ ]:
#01 — Load the manifest, filter to RGB
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path("/content/drive/MyDrive/vip_cup_project/manifests/image_label_manifest.csv")
manifest_df = pd.read_csv(MANIFEST_PATH)

rgb_df = manifest_df[manifest_df["modality"] == "RGB"].copy()
print(rgb_df.groupby("split").size())


In [ ]:
#02 Build the YOLO-format dataset directory
import os
import random

QUICK_TEST = True
QUICK_TRAIN_SIZE = 3000
QUICK_VAL_SIZE = 800

YOLO_ROOT = Path("/content/yolo_rgb_dataset")

random.seed(42)


def build_yolo_split(split_df, split_name, root):
    img_dir = root / "images" / split_name
    lbl_dir = root / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for _, row in split_df.iterrows():
        img_src = Path(row["image_path"])
        lbl_src = Path(row["label_path"])

        img_dst = img_dir / img_src.name
        lbl_dst = lbl_dir / lbl_src.name

        if not img_dst.exists():
            os.symlink(img_src, img_dst)
        if not lbl_dst.exists():
            os.symlink(lbl_src, lbl_dst)

    return len(split_df)


split_sizes = {}
for split_name in ["train", "val", "test"]:
    split_df = rgb_df[rgb_df["split"] == split_name]

    if QUICK_TEST:
        n = QUICK_TRAIN_SIZE if split_name == "train" else QUICK_VAL_SIZE
        split_df = split_df.sample(n=min(n, len(split_df)), random_state=42)

    split_sizes[split_name] = build_yolo_split(split_df, split_name, YOLO_ROOT)

print("Dataset built at:", YOLO_ROOT)
print(split_sizes)
if QUICK_TEST:
    print("QUICK_TEST is ON — this is a subset. Set QUICK_TEST = False for the real training run.")


In [ ]:
DATA_YAML_PATH = YOLO_ROOT / "data.yaml"

data_yaml_content = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: bird
  1: drone
"""

DATA_YAML_PATH.write_text(data_yaml_content)
print(data_yaml_content)


In [ ]:
#03 — Train YOLOv8n
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4) before training.")


In [ ]:
!pip -q install ultralytics

from ultralytics import YOLO

EPOCHS = 5 if QUICK_TEST else 30
IMG_SIZE = 384
BATCH_SIZE = 32
RUN_NAME = "model_rgb_quicktest" if QUICK_TEST else "model_rgb"

RUNS_DIR = Path("/content/drive/MyDrive/vip_cup_project/runs")

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=10,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_WEIGHTS_PATH = Path(train_results.save_dir) / "weights" / "best.pt"
print("Best weights saved to:", BEST_WEIGHTS_PATH)


In [ ]:
#04 — Evaluate on val and test
eval_model = YOLO(str(BEST_WEIGHTS_PATH))

val_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="val")
test_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="test")

class_names = eval_model.names
print("Classes:", class_names)


In [ ]:
def summarize_metrics(metrics, label):
    p = metrics.box.p   # precision per class
    r = metrics.box.r   # recall per class
    f1 = metrics.box.f1  # F1 per class
    map50 = metrics.box.map50
    map5095 = metrics.box.map

    rows = []
    for i, name in class_names.items():
        rows.append({
            "split": label,
            "class": name,
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
        })

    df = pd.DataFrame(rows)
    print(f"=== {label} ===")
    print(df.to_string(index=False))
    print(f"mAP@0.5: {map50:.4f}   mAP@0.5:0.95: {map5095:.4f}")
    print()
    return df, float(map50), float(map5095)


val_class_df, val_map50, val_map5095 = summarize_metrics(val_metrics, "val")
test_class_df, test_map50, test_map5095 = summarize_metrics(test_metrics, "test")


In [ ]:
#05 Log to the shared model-comparison table
RESULTS_PATH = Path("/content/drive/MyDrive/vip_cup_project/results/model_comparison.csv")

overall_precision = val_class_df["precision"].mean()
overall_recall = val_class_df["recall"].mean()
overall_f1 = val_class_df["f1"].mean()

new_row = {
    "model_name": "Model 2: YOLOv8n" + (" (quick test subset)" if QUICK_TEST else ""),
    "modality": "RGB",
    "accuracy": val_map50,
    "precision": overall_precision,
    "recall": overall_recall,
    "f1": overall_f1,
    "notes": "Accuracy = mAP@0.5 (see notebook intro for why). Precision/Recall/F1 macro-averaged over bird/drone.",
}

if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    results_df = results_df[
        ~((results_df["model_name"] == new_row["model_name"]) & (results_df["modality"] == new_row["modality"]))
    ]
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
else:
    results_df = pd.DataFrame([new_row])

results_df.to_csv(RESULTS_PATH, index=False)
print(results_df.to_string(index=False))


In [ ]:
#06 Record the weights path for later notebooks
import json

MODEL_PATHS_FILE = Path("/content/drive/MyDrive/vip_cup_project/results/model_paths.json")

if MODEL_PATHS_FILE.exists():
    model_paths = json.loads(MODEL_PATHS_FILE.read_text())
else:
    model_paths = {}

model_paths["rgb"] = str(BEST_WEIGHTS_PATH)
MODEL_PATHS_FILE.write_text(json.dumps(model_paths, indent=2))

print("Saved model paths:")
print(json.dumps(model_paths, indent=2))


# 03 — Model 2 (YOLOv8), IR-only

Same structure as `02_model_rgb.ipynb`, trained on the IR split instead. **Confirm
`CUDA available: True` before running this** — the RGB run showed what CPU training
costs (1.5 hours for 5 epochs on 3,000 images), and that applies here too.

In [ ]:
#01 — Load the manifest, filter to IR
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path("/content/drive/MyDrive/vip_cup_project/manifests/image_label_manifest.csv")
manifest_df = pd.read_csv(MANIFEST_PATH)

ir_df = manifest_df[manifest_df["modality"] == "IR"].copy()
print(ir_df.groupby("split").size())


In [ ]:
#02 — Build the YOLO-format dataset directory (IR)
import os
import random

QUICK_TEST = True
QUICK_TRAIN_SIZE = 3000
QUICK_VAL_SIZE = 800

YOLO_ROOT = Path("/content/yolo_ir_dataset")

random.seed(42)


def build_yolo_split(split_df, split_name, root):
    img_dir = root / "images" / split_name
    lbl_dir = root / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for _, row in split_df.iterrows():
        img_src = Path(row["image_path"])
        lbl_src = Path(row["label_path"])

        img_dst = img_dir / img_src.name
        lbl_dst = lbl_dir / lbl_src.name

        if not img_dst.exists():
            os.symlink(img_src, img_dst)
        if not lbl_dst.exists():
            os.symlink(lbl_src, lbl_dst)

    return len(split_df)


split_sizes = {}
for split_name in ["train", "val", "test"]:
    split_df = ir_df[ir_df["split"] == split_name]

    if QUICK_TEST:
        n = QUICK_TRAIN_SIZE if split_name == "train" else QUICK_VAL_SIZE
        split_df = split_df.sample(n=min(n, len(split_df)), random_state=42)

    split_sizes[split_name] = build_yolo_split(split_df, split_name, YOLO_ROOT)

print("Dataset built at:", YOLO_ROOT)
print(split_sizes)
if QUICK_TEST:
    print("QUICK_TEST is ON — this is a subset. Set QUICK_TEST = False for the real training run.")


In [ ]:
DATA_YAML_PATH = YOLO_ROOT / "data.yaml"

data_yaml_content = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: bird
  1: drone
"""

DATA_YAML_PATH.write_text(data_yaml_content)
print(data_yaml_content)


In [ ]:
#03 — Train YOLOv8n on IR by changing GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4) before training.")


In [ ]:
!pip -q install ultralytics

from ultralytics import YOLO

EPOCHS = 5 if QUICK_TEST else 30
IMG_SIZE = 384
BATCH_SIZE = 32
RUN_NAME = "model_ir_quicktest" if QUICK_TEST else "model_ir"

RUNS_DIR = Path("/content/drive/MyDrive/vip_cup_project/runs")

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=10,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_WEIGHTS_PATH = Path(train_results.save_dir) / "weights" / "best.pt"
print("Best weights saved to:", BEST_WEIGHTS_PATH)


In [ ]:
#04 — Evaluate on val and test
eval_model = YOLO(str(BEST_WEIGHTS_PATH))

val_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="val")
test_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="test")

class_names = eval_model.names
print("Classes:", class_names)


In [ ]:
def summarize_metrics(metrics, label):
    p = metrics.box.p
    r = metrics.box.r
    f1 = metrics.box.f1
    map50 = metrics.box.map50
    map5095 = metrics.box.map

    rows = []
    for i, name in class_names.items():
        rows.append({
            "split": label,
            "class": name,
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
        })

    df = pd.DataFrame(rows)
    print(f"=== {label} ===")
    print(df.to_string(index=False))
    print(f"mAP@0.5: {map50:.4f}   mAP@0.5:0.95: {map5095:.4f}")
    print()
    return df, float(map50), float(map5095)


val_class_df, val_map50, val_map5095 = summarize_metrics(val_metrics, "val")
test_class_df, test_map50, test_map5095 = summarize_metrics(test_metrics, "test")


In [ ]:
#05 — Log to the shared model-comparison table
RESULTS_PATH = Path("/content/drive/MyDrive/vip_cup_project/results/model_comparison.csv")

overall_precision = val_class_df["precision"].mean()
overall_recall = val_class_df["recall"].mean()
overall_f1 = val_class_df["f1"].mean()

new_row = {
    "model_name": "Model 2: YOLOv8n" + (" (quick test subset)" if QUICK_TEST else ""),
    "modality": "IR",
    "accuracy": val_map50,
    "precision": overall_precision,
    "recall": overall_recall,
    "f1": overall_f1,
    "notes": "Accuracy = mAP@0.5 (see 02_model_rgb.ipynb intro for why). Precision/Recall/F1 macro-averaged over bird/drone.",
}

if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    results_df = results_df[
        ~((results_df["model_name"] == new_row["model_name"]) & (results_df["modality"] == new_row["modality"]))
    ]
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
else:
    results_df = pd.DataFrame([new_row])

results_df.to_csv(RESULTS_PATH, index=False)
print(results_df.to_string(index=False))


In [ ]:
#06 — Record the weights path for later notebooks
import json

MODEL_PATHS_FILE = Path("/content/drive/MyDrive/vip_cup_project/results/model_paths.json")

if MODEL_PATHS_FILE.exists():
    model_paths = json.loads(MODEL_PATHS_FILE.read_text())
else:
    model_paths = {}

model_paths["ir"] = str(BEST_WEIGHTS_PATH)
MODEL_PATHS_FILE.write_text(json.dumps(model_paths, indent=2))

print("Saved model paths:")
print(json.dumps(model_paths, indent=2))


 4 — Model 2 (YOLOv8), RGB+IR Early Fusion

**Goal:** train the third required model — a single YOLOv8n detector that takes a **4-channel**

In [ ]:
#01 — Install dependencies, mount Drive, load the fusion manifest
!pip -q install opencv-python pandas tqdm pillow ultralytics

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vip_cup_project")
FUSION_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "fusion_pairs_manifest.csv"

assert FUSION_MANIFEST_PATH.exists(), f"Missing {FUSION_MANIFEST_PATH} — run 00_setup_and_data.ipynb first."

fusion_df = pd.read_csv(FUSION_MANIFEST_PATH)
print(f"Loaded fusion manifest: {len(fusion_df):,} matched RGB/IR pairs")
print(fusion_df.groupby("split").size())

In [ ]:
#02 — Spot-check: do the RGB and IR label files agree for a sample of pairs?
# Early fusion assumes one shared ground-truth per stem. If RGB and IR labels
# disagree a lot, that's worth knowing before training (would mean the two
# modalities were annotated somewhat independently). We use the RGB label as
# the fused label going forward either way, but this check flags surprises.
import random
random.seed(42)

SPOT_CHECK_N = 300
sample_rows = fusion_df.sample(n=min(SPOT_CHECK_N, len(fusion_df)), random_state=42)

mismatches = 0
for _, row in sample_rows.iterrows():
    rgb_text = Path(row["rgb_label_path"]).read_text().strip()
    ir_text = Path(row["ir_label_path"]).read_text().strip()
    if rgb_text != ir_text:
        mismatches += 1

print(f"RGB/IR label mismatches in sample: {mismatches}/{len(sample_rows)} "
      f"({100 * mismatches / len(sample_rows):.1f}%)")
print("Using the RGB label file as the fused ground truth for all pairs.")

In [ ]:
#03 — Build the fused 4-channel .npy dataset + YOLO-format directory
import os
import shutil
import cv2
import numpy as np
from tqdm import tqdm

QUICK_TEST = True
QUICK_TRAIN_SIZE = 3000
QUICK_VAL_SIZE = 800

YOLO_ROOT = Path("/content/yolo_fusion_dataset")
random.seed(42)

def build_fusion_split(split_df, split_name, root):
    img_dir = root / "images" / split_name
    lbl_dir = root / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    n_written = 0
    n_skipped = 0
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Building {split_name} fusion npy"):
        stem = row["stem"]
        npy_path = img_dir / f"{stem}.npy"
        lbl_dst = lbl_dir / f"{stem}.txt"

        if not npy_path.exists():
            rgb = cv2.imread(str(row["rgb_image_path"]), cv2.IMREAD_COLOR)   # H x W x 3, BGR
            ir = cv2.imread(str(row["ir_image_path"]), cv2.IMREAD_GRAYSCALE)  # H x W
            if rgb is None or ir is None:
                n_skipped += 1
                continue
            if ir.shape[:2] != rgb.shape[:2]:
                ir = cv2.resize(ir, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
            fused = np.dstack([rgb, ir]).astype(np.uint8)  # H x W x 4 (B, G, R, IR)
            np.save(npy_path, fused)

        if not lbl_dst.exists():
            shutil.copy(row["rgb_label_path"], lbl_dst)

        n_written += 1
    return n_written, n_skipped

split_sizes = {}
for split_name in ["train", "val", "test"]:
    split_df = fusion_df[fusion_df["split"] == split_name]
    if QUICK_TEST:
        n = QUICK_TRAIN_SIZE if split_name == "train" else QUICK_VAL_SIZE
        split_df = split_df.sample(n=min(n, len(split_df)), random_state=42)
    written, skipped = build_fusion_split(split_df, split_name, YOLO_ROOT)
    split_sizes[split_name] = written
    if skipped:
        print(f"  {split_name}: skipped {skipped} pairs (unreadable image)")

print("Dataset built at:", YOLO_ROOT)
print(split_sizes)
if QUICK_TEST:
    print("QUICK_TEST is ON — this is a subset. Set QUICK_TEST = False for the real training run.")

In [ ]:
DATA_YAML_PATH = YOLO_ROOT / "data.yaml"
data_yaml_content = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test
nc: 2
channels: 4
names:
  0: bird
  1: drone
"""
DATA_YAML_PATH.write_text(data_yaml_content)
print(data_yaml_content)

In [ ]:
#04 — Monkey-patch Ultralytics for 4-channel .npy loading
import math
import ultralytics.data.utils as ultra_utils
import ultralytics.data.base as ultra_base
import ultralytics.data.dataset as ultra_dataset
import ultralytics.data.build as ultra_build
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import verify_image_label as _orig_verify_image_label
from ultralytics.utils import colorstr

# Let the file scanner recognize .npy as an image file
NEW_IMG_FORMATS = frozenset(list(ultra_utils.IMG_FORMATS) + ["npy"])
for mod in (ultra_utils, ultra_base, ultra_dataset):
    if hasattr(mod, "IMG_FORMATS"):
        mod.IMG_FORMATS = NEW_IMG_FORMATS

# --- Patch #1: pixel loading during training (this part was already correct) ---
class FusionYOLODataset(YOLODataset):
    """YOLODataset that loads pre-fused 4-channel .npy arrays instead of
    calling cv2.imread. Everything else (labels, augmentation pipeline,
    caching) is inherited unchanged."""

    def load_image(self, i, rect_mode=True):
        f = self.im_files[i]
        im = np.load(f)  # H x W x 4, uint8
        h0, w0 = im.shape[:2]
        if rect_mode:
            r = self.imgsz / max(h0, w0)
            if r != 1:
                interp = cv2.INTER_LINEAR if (self.augment or r > 1) else cv2.INTER_AREA
                w, h = (min(math.ceil(w0 * r), self.imgsz), min(math.ceil(h0 * r), self.imgsz))
                im = cv2.resize(im, (w, h), interpolation=interp)
        elif (h0, w0) != (self.imgsz, self.imgsz):
            im = cv2.resize(im, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)
        if im.ndim == 2:
            im = im[..., None]

        self.ims[i], self.im_hw0[i], self.im_hw[i] = im, (h0, w0), im.shape[:2]
        self.buffer.append(i)
        if len(self.buffer) >= self.max_buffer_length:
            j = self.buffer.pop(0)
            if not self.cache_ram:
                self.ims[j] = None
        return im, (h0, w0), im.shape[:2]

def build_fusion_yolo_dataset(cfg, img_path, batch, data, mode="train", rect=False, stride=32, multi_modal=False):
    return FusionYOLODataset(
        img_path=img_path,
        imgsz=cfg.imgsz,
        batch_size=batch,
        augment=mode == "train",
        hyp=cfg,
        rect=cfg.rect or rect,
        cache=cfg.cache or None,
        single_cls=cfg.single_cls or False,
        stride=int(stride),
        pad=0.0 if mode == "train" else 0.5,
        prefix=colorstr(f"{mode}: "),
        task=cfg.task,
        classes=cfg.classes,
        data=data,
        fraction=cfg.fraction if mode == "train" else 1.0,
    )

ultra_build.build_yolo_dataset = build_fusion_yolo_dataset

# --- Patch #2: label-caching / file verification pass (this was the missing piece) ---
# Ultralytics runs a separate pre-pass (cache_labels -> verify_image_label) that opens
# every image with PIL to check it's valid and read its shape, BEFORE any training
# starts. PIL can't open .npy files, so every image was being marked corrupt and
# discarded, leaving zero labels. This replaces that check with a numpy-based one
# for .npy files, and falls back to the original PIL-based check for anything else.
def fusion_verify_image_label(args):
    # Index positionally instead of unpacking the full tuple — the number of
    # elements Ultralytics passes here has changed between versions, but
    # (im_file, lb_file, prefix) as the first three is stable across versions.
    im_file, lb_file, prefix = args[0], args[1], args[2]
    if not str(im_file).lower().endswith(".npy"):
        return _orig_verify_image_label(args)

    nm, nf, ne, nc, msg, segments, keypoints = 0, 0, 0, 0, "", [], None
    try:
        arr = np.load(im_file, mmap_mode="r")
        h, w = arr.shape[0], arr.shape[1]
        shape = (h, w)
        assert (shape[0] > 9) and (shape[1] > 9), f"image size {shape} <10 pixels"

        if os.path.isfile(lb_file):
            nf = 1
            with open(lb_file) as f:
                lb = [x.split() for x in f.read().strip().splitlines() if len(x)]
                lb = np.array(lb, dtype=np.float32)
            nl = len(lb)
            if nl:
                assert lb.shape[1] == 5, f"labels require 5 columns, {lb.shape[1]} columns detected"
                assert lb[:, 1:].max() <= 1, f"non-normalized or out of bounds coordinates"
                assert lb.min() >= 0, f"negative label values"
                _, i = np.unique(lb, axis=0, return_index=True)
                if len(i) < nl:
                    lb = lb[i]
                    msg = f"{prefix}WARNING \u26a0\ufe0f {im_file}: {nl - len(i)} duplicate labels removed"
            else:
                ne = 1
                lb = np.zeros((0, 5), dtype=np.float32)
        else:
            nm = 1
            lb = np.zeros((0, 5), dtype=np.float32)

        return im_file, lb, shape, segments, keypoints, nm, nf, ne, nc, msg
    except Exception as e:
        nc = 1
        msg = f"{prefix}WARNING \u26a0\ufe0f {im_file}: ignoring corrupt image/label: {e}"
        return [None, None, None, None, None, nm, nf, ne, nc, msg]

ultra_dataset.verify_image_label = fusion_verify_image_label

print("Ultralytics patched for 4-channel .npy loading (pixel loading + label verification).")

In [ ]:
#05 — Load YOLOv8n and expand the first conv layer from 3 → 4 input channels
import torch
import torch.nn as nn
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4) before training.")

def convert_first_conv_to_4ch(model):
    """Replace YOLOv8's first conv (3 in-channels) with a 4 in-channel conv.
    Pretrained RGB weights are copied into the first 3 channels; the 4th
    (IR) channel is initialized as the mean of the RGB filters — a standard
    trick for extending pretrained conv filters to an extra input channel."""
    first_conv = model.model.model[0].conv
    old_weight = first_conv.weight.data  # (out_ch, 3, k, k)

    new_conv = nn.Conv2d(
        in_channels=4,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None,
    )
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = old_weight
        new_conv.weight[:, 3:4, :, :] = old_weight.mean(dim=1, keepdim=True)
        if first_conv.bias is not None:
            new_conv.bias[:] = first_conv.bias.data

    model.model.model[0].conv = new_conv
    if hasattr(model.model, "yaml") and isinstance(model.model.yaml, dict):
        model.model.yaml["ch"] = 4
    return model

model = YOLO("yolov8n.pt")
model = convert_first_conv_to_4ch(model)
print("First conv layer:", model.model.model[0].conv)

In [ ]:
#06 — Train
EPOCHS = 5 if QUICK_TEST else 30
IMG_SIZE = 384
BATCH_SIZE = 32
RUN_NAME = "model_fusion_quicktest" if QUICK_TEST else "model_fusion"
RUNS_DIR = Path("/content/drive/MyDrive/vip_cup_project/runs")

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=10,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,  # HSV jitter assumes 3-ch BGR — must be off for 4-ch input
)

BEST_WEIGHTS_PATH = Path(train_results.save_dir) / "weights" / "best.pt"
print("Best weights saved to:", BEST_WEIGHTS_PATH)

In [ ]:
#07 — Evaluate on val and test
eval_model = YOLO(str(BEST_WEIGHTS_PATH))  # checkpoint stores the modified 4-ch architecture, restored automatically

val_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="val")
test_metrics = eval_model.val(data=str(DATA_YAML_PATH), split="test")

class_names = eval_model.names
print("Classes:", class_names)

In [ ]:
def summarize_metrics(metrics, label):
    p = metrics.box.p
    r = metrics.box.r
    f1 = metrics.box.f1
    map50 = metrics.box.map50
    map5095 = metrics.box.map
    rows = []
    for i, name in class_names.items():
        rows.append({
            "split": label,
            "class": name,
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
        })
    df = pd.DataFrame(rows)
    print(f"=== {label} ===")
    print(df.to_string(index=False))
    print(f"mAP@0.5: {map50:.4f}   mAP@0.5:0.95: {map5095:.4f}")
    print()
    return df, float(map50), float(map5095)

val_class_df, val_map50, val_map5095 = summarize_metrics(val_metrics, "val")
test_class_df, test_map50, test_map5095 = summarize_metrics(test_metrics, "test")

In [ ]:
#08 — Log to the shared model-comparison table
RESULTS_PATH = Path("/content/drive/MyDrive/vip_cup_project/results/model_comparison.csv")

overall_precision = val_class_df["precision"].mean()
overall_recall = val_class_df["recall"].mean()
overall_f1 = val_class_df["f1"].mean()

new_row = {
    "model_name": "Model 2: YOLOv8n (fusion)" + (" (quick test subset)" if QUICK_TEST else ""),
    "modality": "RGB+IR",
    "accuracy": val_map50,
    "precision": overall_precision,
    "recall": overall_recall,
    "f1": overall_f1,
    "notes": "4-channel early fusion (RGB + IR stacked, first conv expanded 3→4ch). Accuracy = mAP@0.5 (see 02_model_rgb.ipynb intro for why). Precision/recall/f1 are mean across classes at IoU≥0.5.",
}

if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    results_df = results_df[
        ~((results_df["model_name"] == new_row["model_name"]) & (results_df["modality"] == new_row["modality"]))
    ]
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
else:
    results_df = pd.DataFrame([new_row])

results_df.to_csv(RESULTS_PATH, index=False)
print(results_df.to_string(index=False))

In [ ]:
#09 — Record the weights path for later notebooks (05 robustness, 06 speed, 07 demo)
import json

MODEL_PATHS_FILE = Path("/content/drive/MyDrive/vip_cup_project/results/model_paths.json")
if MODEL_PATHS_FILE.exists():
    model_paths = json.loads(MODEL_PATHS_FILE.read_text())
else:
    model_paths = {}

model_paths["fusion"] = str(BEST_WEIGHTS_PATH)
MODEL_PATHS_FILE.write_text(json.dumps(model_paths, indent=2))
print("Saved model paths:")
print(json.dumps(model_paths, indent=2))
print()
print("Re-run 05_robustness_testing.ipynb and 06_speed_benchmarking.ipynb now —")
print("they check model_paths.json for a 'fusion' key and will pick it up automatically.")

# 5 — Robustness Testing

**Goal:** measure how much detection performance degrades under four realistic sensor/transmission degradations — **Gaussian blur, sensor noise, brightness shift, and JPEG compression**

In [ ]:
#01 — Mount Drive, load model paths and manifest
from google.colab import drive
drive.mount("/content/drive")

import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vip_cup_project")
MODEL_PATHS_FILE = PROJECT_ROOT / "results" / "model_paths.json"
MANIFEST_PATH = PROJECT_ROOT / "manifests" / "image_label_manifest.csv"
FUSION_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "fusion_pairs_manifest.csv"

assert MODEL_PATHS_FILE.exists(), f"Missing {MODEL_PATHS_FILE} — run 02 and 03 first."
model_paths = json.loads(MODEL_PATHS_FILE.read_text())
print("Available trained models:", model_paths)

manifest_df = pd.read_csv(MANIFEST_PATH)

HAS_FUSION = "fusion" in model_paths
if not HAS_FUSION:
    print()
    print("NOTE: no 'fusion' entry in model_paths.json yet — 04_model_fusion.ipynb")
    print("hasn't been run (or hasn't saved its weights path). This notebook will")
    print("test RGB and IR only. Rerun this notebook after fusion training to add it.")


In [ ]:
#02 — Install deps and sample val images per modality
!pip -q install ultralytics opencv-python

import cv2
import numpy as np
import random
from ultralytics import YOLO

random.seed(42)

ROBUSTNESS_SAMPLE_SIZE = 300  # images per modality, kept modest since each severity level re-runs inference

val_rgb_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == "RGB")]
val_ir_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == "IR")]

rgb_sample = val_rgb_df.sample(n=min(ROBUSTNESS_SAMPLE_SIZE, len(val_rgb_df)), random_state=42)
ir_sample = val_ir_df.sample(n=min(ROBUSTNESS_SAMPLE_SIZE, len(val_ir_df)), random_state=42)

print(f"RGB val sample: {len(rgb_sample)}")
print(f"IR val sample:  {len(ir_sample)}")


In [ ]:
#03 — Build a self-contained perturbed YOLO eval set per (modality, perturbation, severity)
# Strategy: write perturbed copies of the sampled images + their original labels into a
# throwaway YOLO-format dir, then call model.val() on that dir. This reuses Ultralytics'
# own mAP/precision/recall computation instead of hand-rolling matching logic.

import shutil
import os

PERTURB_ROOT = Path("/content/robustness_eval")

def apply_blur(img, severity):
    # severity 0..4 -> kernel size 1,3,5,7,9
    k = severity * 2 + 1
    if k <= 1:
        return img
    return cv2.GaussianBlur(img, (k, k), 0)

def apply_noise(img, severity):
    # severity 0..4 -> gaussian noise std as fraction of 255
    std = severity * 12.0
    if std <= 0:
        return img
    noise = np.random.normal(0, std, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def apply_brightness(img, severity):
    # severity -2..2 -> multiplicative brightness shift; here we use 0..4 mapped to factors
    factors = [1.0, 0.7, 0.5, 1.3, 1.6]
    factor = factors[severity]
    out = img.astype(np.float32) * factor
    return np.clip(out, 0, 255).astype(np.uint8)

def apply_jpeg(img, severity):
    # severity 0..4 -> JPEG quality 100,60,40,20,10
    qualities = [100, 60, 40, 20, 10]
    q = qualities[severity]
    ok, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    if not ok:
        return img
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

PERTURBATIONS = {
    "blur": apply_blur,
    "noise": apply_noise,
    "brightness": apply_brightness,
    "jpeg": apply_jpeg,
}
SEVERITY_LEVELS = [0, 1, 2, 3, 4]  # 0 = clean baseline for that perturbation function

def build_perturbed_split(sample_df, modality, perturb_name, severity, root):
    img_dir = root / "images" / "val"
    lbl_dir = root / "labels" / "val"
    if img_dir.exists():
        shutil.rmtree(img_dir)
    if lbl_dir.exists():
        shutil.rmtree(lbl_dir)
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    fn = PERTURBATIONS[perturb_name]
    for _, row in sample_df.iterrows():
        img = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            continue
        perturbed = fn(img, severity)
        out_name = Path(row["image_path"]).name
        cv2.imwrite(str(img_dir / out_name), perturbed)
        lbl_src = Path(row["label_path"])
        lbl_dst = lbl_dir / lbl_src.with_suffix(".txt").name
        if not lbl_dst.exists():
            shutil.copy(lbl_src, lbl_dst)

    data_yaml = root / "data.yaml"
    data_yaml.write_text(
        f"""path: {root}
train: images/val
val: images/val
test: images/val
nc: 2
names:
  0: bird
  1: drone
"""
    )
    return data_yaml

print("Perturbation + eval-set builder ready.")


In [ ]:
#04 — Run the robustness sweep for each available model
results_rows = []

MODALITY_SAMPLES = {"rgb": rgb_sample, "ir": ir_sample}

for modality_key, sample_df in MODALITY_SAMPLES.items():
    if modality_key not in model_paths:
        print(f"Skipping {modality_key}: no trained weights in model_paths.json")
        continue

    weights_path = model_paths[modality_key]
    model = YOLO(weights_path)
    print(f"\n=== Modality: {modality_key.upper()} — weights: {weights_path} ===")

    for perturb_name in PERTURBATIONS:
        for severity in SEVERITY_LEVELS:
            eval_root = PERTURB_ROOT / modality_key / perturb_name / str(severity)
            data_yaml = build_perturbed_split(sample_df, modality_key, perturb_name, severity, eval_root)

            metrics = model.val(data=str(data_yaml), split="val", verbose=False)

            results_rows.append({
                "modality": modality_key,
                "perturbation": perturb_name,
                "severity": severity,
                "map50": float(metrics.box.map50),
                "map50_95": float(metrics.box.map),
                "precision": float(metrics.box.p.mean()),
                "recall": float(metrics.box.r.mean()),
                "f1": float(metrics.box.f1.mean()),
            })
            print(f"  {perturb_name:10s} severity={severity}  mAP50={metrics.box.map50:.3f}")

robustness_df = pd.DataFrame(results_rows)
print()
print(robustness_df.to_string(index=False))


In [ ]:
#05 — Plot degradation curves (metric vs. severity, one chart per perturbation)
import matplotlib.pyplot as plt

if len(robustness_df) > 0:
    for perturb_name in PERTURBATIONS:
        subset = robustness_df[robustness_df["perturbation"] == perturb_name]
        if subset.empty:
            continue
        plt.figure(figsize=(6, 4))
        for modality_key, group in subset.groupby("modality"):
            group_sorted = group.sort_values("severity")
            plt.plot(group_sorted["severity"], group_sorted["map50"], marker="o", label=modality_key.upper())
        plt.title(f"mAP@0.5 vs. {perturb_name} severity")
        plt.xlabel("Severity level")
        plt.ylabel("mAP@0.5")
        plt.ylim(0, 1)
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plot_path = PROJECT_ROOT / "results" / f"robustness_{perturb_name}.png"
        plt.savefig(plot_path, dpi=120)
        plt.show()
        print("Saved plot:", plot_path)
else:
    print("No results to plot — check that model_paths.json has 'rgb' and/or 'ir' entries.")


In [ ]:
#06 — Save results CSV and log a summary row per model to model_comparison.csv
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

robustness_csv_path = RESULTS_DIR / "robustness_results.csv"
robustness_df.to_csv(robustness_csv_path, index=False)
print("Saved full robustness results to:", robustness_csv_path)

# Summary: average relative mAP drop from clean (severity 0) to worst severity (4), per modality
summary_rows = []
for modality_key, group in robustness_df.groupby("modality"):
    clean_map = group[group["severity"] == 0]["map50"].mean()
    worst_map = group[group["severity"] == group["severity"].max()]["map50"].mean()
    rel_drop = (clean_map - worst_map) / clean_map if clean_map > 0 else float("nan")
    summary_rows.append({
        "modality": modality_key,
        "clean_map50": clean_map,
        "worst_case_map50": worst_map,
        "relative_drop_pct": rel_drop * 100,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

MODEL_COMPARISON_PATH = RESULTS_DIR / "model_comparison.csv"
comparison_rows = []
for _, r in summary_df.iterrows():
    comparison_rows.append({
        "model_name": f"Robustness sweep ({r['modality'].upper()})",
        "modality": r["modality"].upper(),
        "accuracy": r["worst_case_map50"],
        "precision": None,
        "recall": None,
        "f1": None,
        "notes": f"Worst-case mAP@0.5 across blur/noise/brightness/jpeg severities 0-4. "
                 f"Clean mAP50={r['clean_map50']:.3f}, relative drop={r['relative_drop_pct']:.1f}%.",
    })

new_rows_df = pd.DataFrame(comparison_rows)
if MODEL_COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(MODEL_COMPARISON_PATH)
    comparison_df = comparison_df[~comparison_df["model_name"].isin(new_rows_df["model_name"])]
    comparison_df = pd.concat([comparison_df, new_rows_df], ignore_index=True)
else:
    comparison_df = new_rows_df

comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)
print()
print("Logged summary rows to model_comparison.csv")
print(comparison_df.to_string(index=False))


# 6 — Speed Benchmarking

**Goal:** measure real-world inference speed for each trained model (RGB, IR, and Fusion once available) — the rubric's "speed benchmarking" requirement.

In [ ]:
#01 — Mount Drive, load model paths and manifest
from google.colab import drive
drive.mount("/content/drive")

import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vip_cup_project")
MODEL_PATHS_FILE = PROJECT_ROOT / "results" / "model_paths.json"
MANIFEST_PATH = PROJECT_ROOT / "manifests" / "image_label_manifest.csv"

assert MODEL_PATHS_FILE.exists(), f"Missing {MODEL_PATHS_FILE} — run 02 and 03 first."
model_paths = json.loads(MODEL_PATHS_FILE.read_text())
print("Available trained models:", model_paths)

manifest_df = pd.read_csv(MANIFEST_PATH)

HAS_FUSION = "fusion" in model_paths
if not HAS_FUSION:
    print()
    print("NOTE: no 'fusion' entry in model_paths.json yet — this notebook will")
    print("benchmark RGB and IR only. Rerun after 04_model_fusion.ipynb saves weights.")


In [ ]:
#02 — Confirm hardware and install deps
import torch
!pip -q install ultralytics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: benchmarking on CPU. Numbers here are NOT comparable to a GPU")
    print("deployment target. Switch Runtime > Change runtime type > T4 GPU for")
    print("numbers that reflect realistic edge/server deployment, then rerun.")


In [ ]:
#03 — Sample a fixed set of images per modality for timing
import random
random.seed(42)

TIMING_SAMPLE_SIZE = 200  # images used to build timing batches
WARMUP_ITERS = 10
TIMED_ITERS = 100
BATCH_SIZES = [1, 16]

val_rgb_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == "RGB")]
val_ir_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == "IR")]

rgb_sample_paths = val_rgb_df.sample(n=min(TIMING_SAMPLE_SIZE, len(val_rgb_df)), random_state=42)["image_path"].tolist()
ir_sample_paths = val_ir_df.sample(n=min(TIMING_SAMPLE_SIZE, len(val_ir_df)), random_state=42)["image_path"].tolist()

MODALITY_IMAGE_PATHS = {"rgb": rgb_sample_paths, "ir": ir_sample_paths}
print({k: len(v) for k, v in MODALITY_IMAGE_PATHS.items()})


In [ ]:
#04 — Timing helper: warmup + repeated inference, batch size aware
import time
import itertools
from ultralytics import YOLO

def benchmark_model(weights_path, image_paths, batch_size, warmup_iters=WARMUP_ITERS, timed_iters=TIMED_ITERS, device=DEVICE):
    model = YOLO(weights_path)

    # Build a repeating cycle of image paths so we always have enough for warmup + timed batches
    cycler = itertools.cycle(image_paths)

    def next_batch():
        return [next(cycler) for _ in range(batch_size)]

    # Warmup — first few calls include CUDA kernel compilation / cache warming, exclude from timing
    for _ in range(warmup_iters):
        model.predict(next_batch(), device=device, verbose=False)

    # Timed loop
    latencies_ms = []
    for _ in range(timed_iters):
        batch = next_batch()
        start = time.perf_counter()
        model.predict(batch, device=device, verbose=False)
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        latencies_ms.append(elapsed_ms / batch_size)  # normalize to per-image ms

    latencies_ms = sorted(latencies_ms)
    n = len(latencies_ms)
    return {
        "mean_ms_per_image": sum(latencies_ms) / n,
        "median_ms_per_image": latencies_ms[n // 2],
        "p95_ms_per_image": latencies_ms[int(n * 0.95)],
        "fps": 1000.0 / (sum(latencies_ms) / n),
    }

print("Benchmark helper ready.")


In [ ]:
#05 — Run the benchmark sweep for each available model x batch size
import os

speed_rows = []

for modality_key, image_paths in MODALITY_IMAGE_PATHS.items():
    if modality_key not in model_paths:
        print(f"Skipping {modality_key}: no trained weights in model_paths.json")
        continue

    weights_path = model_paths[modality_key]
    weights_size_mb = os.path.getsize(weights_path) / (1024 * 1024)

    model_for_params = YOLO(weights_path)
    n_params = sum(p.numel() for p in model_for_params.model.parameters())

    print(f"\n=== {modality_key.upper()} — {weights_path} ===")
    print(f"Params: {n_params:,}   File size: {weights_size_mb:.2f} MB")

    for batch_size in BATCH_SIZES:
        result = benchmark_model(weights_path, image_paths, batch_size)
        result.update({
            "modality": modality_key,
            "batch_size": batch_size,
            "device": DEVICE,
            "params": n_params,
            "weights_size_mb": weights_size_mb,
        })
        speed_rows.append(result)
        print(f"  batch={batch_size:3d}  mean={result['mean_ms_per_image']:.2f}ms/img  "
              f"p95={result['p95_ms_per_image']:.2f}ms/img  FPS={result['fps']:.1f}")

speed_df = pd.DataFrame(speed_rows)
print()
print(speed_df.to_string(index=False))


In [ ]:
#06 — Plot latency vs. batch size per model
import matplotlib.pyplot as plt

if len(speed_df) > 0:
    plt.figure(figsize=(6, 4))
    for modality_key, group in speed_df.groupby("modality"):
        group_sorted = group.sort_values("batch_size")
        plt.plot(group_sorted["batch_size"], group_sorted["mean_ms_per_image"], marker="o", label=modality_key.upper())
    plt.title(f"Inference latency vs. batch size ({DEVICE.upper()})")
    plt.xlabel("Batch size")
    plt.ylabel("ms / image")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    RESULTS_DIR = PROJECT_ROOT / "results"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_path = RESULTS_DIR / f"speed_latency_{DEVICE}.png"
    plt.savefig(plot_path, dpi=120)
    plt.show()
    print("Saved plot:", plot_path)
else:
    print("No results to plot — check that model_paths.json has 'rgb' and/or 'ir' entries.")


In [ ]:
#07 — Save results CSV and log a summary row per model to model_comparison.csv
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

speed_csv_path = RESULTS_DIR / "speed_benchmark.csv"
speed_df.to_csv(speed_csv_path, index=False)
print("Saved full speed benchmark results to:", speed_csv_path)

# Summary row per model at batch size 1 (the deployment-realistic number)
MODEL_COMPARISON_PATH = RESULTS_DIR / "model_comparison.csv"
comparison_rows = []
for modality_key, group in speed_df.groupby("modality"):
    bs1 = group[group["batch_size"] == 1]
    if bs1.empty:
        continue
    row = bs1.iloc[0]
    comparison_rows.append({
        "model_name": f"Speed benchmark ({modality_key.upper()})",
        "modality": modality_key.upper(),
        "accuracy": None,
        "precision": None,
        "recall": None,
        "f1": None,
        "notes": f"batch=1, device={row['device']}: {row['mean_ms_per_image']:.2f} ms/img "
                 f"({row['fps']:.1f} FPS), {row['params']:,} params, {row['weights_size_mb']:.1f} MB.",
    })

new_rows_df = pd.DataFrame(comparison_rows)
if MODEL_COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(MODEL_COMPARISON_PATH)
    comparison_df = comparison_df[~comparison_df["model_name"].isin(new_rows_df["model_name"])]
    comparison_df = pd.concat([comparison_df, new_rows_df], ignore_index=True)
else:
    comparison_df = new_rows_df

comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)
print()
print("Logged summary rows to model_comparison.csv")
print(comparison_df.to_string(index=False))


# 7 — Demo Notebook

**Goal:** load the best-performing trained model (auto-selected from `model_comparison.csv`, override-able below), run inference on a handful of sample images from the val split, and visualize predicted boxes against ground truth

In [ ]:
#01 — Mount Drive, load model paths + comparison table
from google.colab import drive
drive.mount("/content/drive")

import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vip_cup_project")
MODEL_PATHS_FILE = PROJECT_ROOT / "results" / "model_paths.json"
COMPARISON_PATH = PROJECT_ROOT / "results" / "model_comparison.csv"
MANIFEST_PATH = PROJECT_ROOT / "manifests" / "image_label_manifest.csv"
FUSION_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "fusion_pairs_manifest.csv"

assert MODEL_PATHS_FILE.exists(), f"Missing {MODEL_PATHS_FILE} — run 02/03/04 first."

model_paths = json.loads(MODEL_PATHS_FILE.read_text())
manifest_df = pd.read_csv(MANIFEST_PATH)
fusion_df = pd.read_csv(FUSION_MANIFEST_PATH) if FUSION_MANIFEST_PATH.exists() else None

print("Available trained models:", model_paths)


In [ ]:
#02 — Pick the model to demo (auto-picks the highest logged mAP@0.5, or set manually)
MODEL_CHOICE = None  # e.g. "fusion", "ir", "rgb" — leave None to auto-pick

if MODEL_CHOICE is None and COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(COMPARISON_PATH)
    yolo_rows = comparison_df[comparison_df["model_name"].str.startswith("Model 2", na=False)]
    yolo_rows = yolo_rows.dropna(subset=["accuracy"])
    if not yolo_rows.empty:
        best_row = yolo_rows.loc[yolo_rows["accuracy"].idxmax()]
        modality_map = {"RGB": "rgb", "IR": "ir", "RGB+IR": "fusion"}
        MODEL_CHOICE = modality_map.get(best_row["modality"], "ir")
        print(f"Auto-picked: {MODEL_CHOICE}  "
              f"(mAP@0.5={best_row['accuracy']:.4f} from '{best_row['model_name']}')")

if MODEL_CHOICE is None:
    MODEL_CHOICE = "ir" if "ir" in model_paths else next(iter(model_paths))
    print(f"No comparison table found — falling back to: {MODEL_CHOICE}")

assert MODEL_CHOICE in model_paths, f"No trained weights for '{MODEL_CHOICE}' in model_paths.json"
WEIGHTS_PATH = model_paths[MODEL_CHOICE]
print("Using weights:", WEIGHTS_PATH)


In [ ]:
#03 — Load the model + inference helpers
!pip -q install ultralytics opencv-python

import cv2
import numpy as np
import torch
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRES = 0.25
IOU_THRES = 0.45
IMG_SIZE = 384
CLASS_NAMES = {0: "bird", 1: "drone"}

model = YOLO(WEIGHTS_PATH)
model.to(DEVICE)
n_in_channels = model.model.model[0].conv.in_channels
print(f"Loaded '{MODEL_CHOICE}' model on {DEVICE} — expects {n_in_channels}-channel input")

def load_input_image(row, modality):
    """Raw uint8 HxWxC array in the same channel layout used at train time (no RGB conversion)."""
    if modality == "fusion":
        rgb = cv2.imread(row["rgb_image_path"], cv2.IMREAD_COLOR)
        ir = cv2.imread(row["ir_image_path"], cv2.IMREAD_GRAYSCALE)
        if ir.shape[:2] != rgb.shape[:2]:
            ir = cv2.resize(ir, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        return np.dstack([rgb, ir]).astype(np.uint8)
    return cv2.imread(row["image_path"], cv2.IMREAD_COLOR)

def run_inference(raw_img):
    # Use the official high-level API to avoid breaking internal imports
    results = model(raw_img, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]

    if len(results.boxes) == 0:
        return np.empty((0, 6))

    # Extract boxes in [x1, y1, x2, y2, conf, cls] format
    boxes = results.boxes.xyxy.cpu().numpy()
    conf = results.boxes.conf.cpu().numpy().reshape(-1, 1)
    cls = results.boxes.cls.cpu().numpy().reshape(-1, 1)

    return np.hstack([boxes, conf, cls])

In [ ]:
#04 — Run inference on a random sample and visualize
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import cv2

DEMO_SAMPLES = 4

# Pick the manifest dataframe based on the modality
if MODEL_CHOICE == "fusion":
    demo_df = fusion_df[fusion_df["split"] == "val"].copy()
else:
    demo_df = manifest_df[(manifest_df["split"] == "val") & (manifest_df["modality"] == MODEL_CHOICE.upper())].copy()

demo_sample = demo_df.sample(n=min(DEMO_SAMPLES, len(demo_df)), random_state=42)

fig, axes = plt.subplots(1, DEMO_SAMPLES, figsize=(20, 5))
if DEMO_SAMPLES == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, demo_sample.iterrows()):
    # 1. Load the raw image suitable for the model
    raw_img = load_input_image(row, MODEL_CHOICE)

    # 2. Run inference
    preds = run_inference(raw_img)

    # 3. For visualization, we need a 3-channel RGB image
    vis_img_path = row["rgb_image_path"] if MODEL_CHOICE == "fusion" else row["image_path"]
    vis_img = cv2.imread(str(vis_img_path), cv2.IMREAD_COLOR)
    vis_img = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)

    ax.imshow(vis_img)
    ax.axis("off")

    title_str = f"File: {Path(vis_img_path).name}\n"

    # Ground Truth boxes (green)
    label_path = row["rgb_label_path"] if MODEL_CHOICE == "fusion" else row["label_path"]
    true_labels = []
    if Path(label_path).exists():
        text = Path(label_path).read_text().strip()
        for line in text.splitlines():
            parts = line.split()
            if len(parts) == 5:
                cid, x_center, y_center, w, h = map(float, parts)
                x1 = (x_center - w / 2) * vis_img.shape[1]
                y1 = (y_center - h / 2) * vis_img.shape[0]
                box_w = w * vis_img.shape[1]
                box_h = h * vis_img.shape[0]
                rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='g', facecolor='none')
                ax.add_patch(rect)
                true_labels.append(CLASS_NAMES.get(int(cid), str(cid)))

    # Predictions (red)
    pred_labels = []
    for p in preds:
        x1, y1, x2, y2, conf, cls = p
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        cname = CLASS_NAMES.get(int(cls), str(cls))
        pred_labels.append(f"{cname} {conf:.2f}")
        ax.text(x1, y1 - 5, f"{cname} {conf:.2f}", color='r', fontsize=10, backgroundcolor="white")

    title_str += f"GT: {', '.join(true_labels)}\nPreds: {', '.join([c.split()[0] for c in pred_labels])}"
    ax.set_title(title_str, fontsize=10)

plt.tight_layout()
plt.show()

print("Green boxes: Ground Truth")
print("Red boxes: Predictions")

In [ ]:
#05 — Display the annotated grid inline
import matplotlib.pyplot as plt
import cv2

# Re-create the 'annotated' list since the previous cell directly plotted them
annotated = []
for _, row in demo_sample.iterrows():
    raw_img = load_input_image(row, MODEL_CHOICE)
    preds = run_inference(raw_img)

    vis_img_path = row["rgb_image_path"] if MODEL_CHOICE == "fusion" else row["image_path"]
    vis_img = cv2.imread(str(vis_img_path), cv2.IMREAD_COLOR)
    stem = Path(vis_img_path).stem

    # Draw ground truth (yellow/green)
    label_path = row["rgb_label_path"] if MODEL_CHOICE == "fusion" else row["label_path"]
    if Path(label_path).exists():
        text = Path(label_path).read_text().strip()
        for line in text.splitlines():
            parts = line.split()
            if len(parts) == 5:
                cid, x_center, y_center, w, h = map(float, parts)
                x1 = int((x_center - w / 2) * vis_img.shape[1])
                y1 = int((y_center - h / 2) * vis_img.shape[0])
                x2 = int((x_center + w / 2) * vis_img.shape[1])
                y2 = int((y_center + h / 2) * vis_img.shape[0])
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 255), 2) # Yellow GT

    # Draw predictions (red for drone, green for bird)
    for p in preds:
        x1, y1, x2, y2, conf, cls = map(float, p)
        color = (0, 0, 255) if int(cls) == 1 else (0, 255, 0) # Red drone, Green bird
        cv2.rectangle(vis_img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(vis_img, f"{CLASS_NAMES.get(int(cls))} {conf:.2f}", (int(x1), max(10, int(y1)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    annotated.append((stem, vis_img, len(preds)))

n = len(annotated)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, (stem, vis, n_dets) in zip(axes, annotated):
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{stem}  ({n_dets} det.)", fontsize=9)
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.suptitle(f"Demo detections — model: {MODEL_CHOICE.upper()}  "
             f"(green=bird pred, red=drone pred, yellow=ground truth)", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
#06 — Save annotated demo images to Drive
DEMO_DIR = PROJECT_ROOT / "demo" / MODEL_CHOICE
DEMO_DIR.mkdir(parents=True, exist_ok=True)

for stem, vis, _ in annotated:
    out_path = DEMO_DIR / f"{stem}_annotated.jpg"
    cv2.imwrite(str(out_path), vis)

print(f"Saved {len(annotated)} annotated demo images to: {DEMO_DIR}")


 8 Final  Report

the results already produced by notebooks `00`–`07` into the official VIP Cup Task 1 submission report — headline Accuracy/F1/Precision/Recall table, baseline-vs-neural comparison, robustness results, speed benchmarks, demo figures, and a documented list of known limitations and design decisions.

In [ ]:
# Mount Drive, load all results tables
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vip_cup_project")
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_COMPARISON_PATH = RESULTS_DIR / "model_comparison.csv"
ROBUSTNESS_PATH = RESULTS_DIR / "robustness_results.csv"
SPEED_PATH = RESULTS_DIR / "speed_benchmark.csv"
CLASS_BALANCE_PATH = PROJECT_ROOT / "manifests" / "class_balance.csv"
MODEL_PATHS_FILE = RESULTS_DIR / "model_paths.json"

for p in [MODEL_COMPARISON_PATH, ROBUSTNESS_PATH, SPEED_PATH, CLASS_BALANCE_PATH, MODEL_PATHS_FILE]:
    assert p.exists(), f"Missing {p} — run the corresponding earlier notebook first."

comparison_df = pd.read_csv(MODEL_COMPARISON_PATH)
robustness_df = pd.read_csv(ROBUSTNESS_PATH)
speed_df = pd.read_csv(SPEED_PATH)
class_balance_df = pd.read_csv(CLASS_BALANCE_PATH)
model_paths = json.loads(MODEL_PATHS_FILE.read_text())

print("Model comparison rows:", len(comparison_df))
print("Robustness rows:", len(robustness_df))
print("Speed rows:", len(speed_df))
print("Trained models available:", list(model_paths.keys()))


In [ ]:
#confirm these are full-training results
quick_test_rows = comparison_df[comparison_df["model_name"].str.contains("quick test", case=False, na=False)]

if not quick_test_rows.empty:
    print("WARNING: comparison table still contains quick-test rows:")
    print(quick_test_rows["model_name"].tolist())
    print()
    print("Re-run 02_model_rgb.ipynb / 03_model_ir.ipynb / 04_model_fusion.ipynb")
    print("with QUICK_TEST = False, then re-run 05 and 06, before finalizing this report.")
else:
    print("No quick-test rows found — comparison table reflects full training runs.")


In [ ]:
# 03 —Results table (Accuracy / F1 / Precision / Recall per modality)
headline = comparison_df[comparison_df["model_name"].str.startswith("Model 2", na=False)].copy()
headline = headline[["model_name", "modality", "accuracy", "precision", "recall", "f1"]]
headline.columns = ["Model", "Modality", "Accuracy (mAP@0.5)", "Precision", "Recall", "F1"]
headline = headline.sort_values("Modality")

print(headline.to_string(index=False))
print()
print("Note on 'Accuracy': Ultralytics/YOLO detection metrics report Precision, Recall,")
print("and F1 per class (matched at IoU>=0.5) but there is no single standard 'accuracy'")
print("scalar for object detection the way there is for classification. mAP@0.5 is used")
print("as the accuracy proxy throughout this project, stated explicitly rather than")
print("silently substituted. If the official VIP Cup rubric provides a scoring script")
print("with a different accuracy definition, swap it in before final submission.")


In [ ]:
# Baseline vs. neural comparison
baseline_row = comparison_df[comparison_df["model_name"].str.contains("Baseline", na=False)]

print("Statistical baseline (non-neural, Model 1 proposals + Gaussian Naive Bayes):")
print(baseline_row[["model_name", "modality", "accuracy", "precision", "recall", "f1"]].to_string(index=False))
print()
print("YOLOv8 (neural) results, same object-only comparison basis:")
print(headline.to_string(index=False))
print()
print("The baseline's low IoU recall on the real dataset (0.02-0.05, despite center-hit")
print("recall of 0.56-1.0) meant its proposal boxes contain the object's center without")
print("tightly bounding it -- a known constraint of the classical method, not a hidden flaw.")
print("This is the empirical basis for using YOLOv8 as the primary detector in this project.")


In [ ]:
#Class balance (from `00_setup_and_data.ipynb`)
# this is the same class-balance table used to infer the `0=bird, 1=drone` mapping by cross-referencing instance counts against a published paper using the same dataset.
print(class_balance_df.to_string(index=False))
print()
totals = class_balance_df.groupby("class_id")["count"].sum()
print("Total instances by class_id across all splits/modalities:")
print(totals.to_string())


In [ ]:
#Robustness summary + degradation plots
summary_rows = []
for modality in sorted(robustness_df["modality"].unique()):
    subset = robustness_df[robustness_df["modality"] == modality]
    clean = subset[subset["severity"] == 0]["map50"].mean()
    worst = subset[subset["severity"] == subset["severity"].max()]["map50"].mean()
    rel_drop = 100 * (clean - worst) / clean if clean > 0 else float("nan")
    summary_rows.append({
        "modality": modality.upper(),
        "clean_map50": round(clean, 4),
        "worst_case_map50": round(worst, 4),
        "relative_drop_pct": round(rel_drop, 1),
    })

robustness_summary_df = pd.DataFrame(summary_rows)
print(robustness_summary_df.to_string(index=False))


In [ ]:
from PIL import Image

for f in sorted(RESULTS_DIR.glob("robustness_*.png")):
    print(f.name)
    display(Image.open(f))


In [ ]:
# Speed benchmark summary
speed_summary = speed_df[speed_df["batch_size"] == 1][
    ["modality", "mean_ms_per_image", "median_ms_per_image", "p95_ms_per_image",
     "fps", "params", "weights_size_mb", "device"]
].copy()
speed_summary = speed_summary.sort_values("modality")

print(speed_summary.to_string(index=False))



In [ ]:
device = speed_df["device"].iloc[0]
plot_path = RESULTS_DIR / f"speed_latency_{device}.png"
if plot_path.exists():
    display(Image.open(plot_path))
else:
    print("Latency plot not found at:", plot_path)


In [ ]:
# Demo figures
demo_dir = PROJECT_ROOT / "demo"

if demo_dir.exists():
    for model_dir in sorted(p for p in demo_dir.iterdir() if p.is_dir()):
        imgs = sorted(model_dir.glob("*_annotated.jpg"))[:4]
        if not imgs:
            continue
        print(f"--- {model_dir.name.upper()} demo detections ---")
        for img_path in imgs:
            display(Image.open(img_path))
else:
    print("No demo/ folder found — run 07_demo.ipynb first.")


In [ ]:
limitations = """
Known limitations and caveats for this submission:

1. Task scope: only Task 1 (drone/bird detection) was completed. Tasks 2 and 3
   were out of scope, since the Kaggle mirror of the VIP Cup dataset lacks the
   video/payload data those tasks require.

2. Class ID mapping (0=bird, 1=drone) was inferred by cross-referencing instance
   counts against a published paper using the same official VIP Cup dataset, not
   from an official data.yaml with named classes. [Insert result of the visual
   spot-check here once performed -- see class_balance table in section 05.]

3. Accuracy substitution: object detection has no single standard "accuracy"
   scalar the way classification does. mAP@0.5 is reported as the accuracy
   figure throughout this project, stated explicitly rather than substituted
   silently. If the official VIP Cup rubric provides a scoring script with a
   different accuracy definition, swap it in before final submission.

4. The statistical baseline (Model 1 proposals + Gaussian Naive Bayes) matches
   proposals to ground truth by center-hit, not strict IoU, because Model 1's
   IoU recall on the real dataset was very low (0.02-0.05) despite high
   center-hit recall (0.56-1.0). This is a known constraint of the classical
   proposal method, not a hidden flaw, and is stated here for transparency.

5. Early fusion assumes RGB and IR frames share a single ground truth per
   filename stem. A spot-check in 04_model_fusion.ipynb found 0 label
   mismatches across a 300-pair sample, supporting this assumption.

6. [Add once full training finishes: any notable differences between the
   quick-test development numbers and the final full-training numbers in
   this report, so a reviewer comparing dev history to the final report
   is not confused by the discrepancy.]
"""
print(limitations)

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "report_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

headline.to_csv(EXPORT_DIR / "headline_results.csv", index=False)
robustness_summary_df.to_csv(EXPORT_DIR / "robustness_summary.csv", index=False)
speed_summary.to_csv(EXPORT_DIR / "speed_summary.csv", index=False)
baseline_row.to_csv(EXPORT_DIR / "baseline_result.csv", index=False)
class_balance_df.to_csv(EXPORT_DIR / "class_balance.csv", index=False)

with open(EXPORT_DIR / "limitations.txt", "w") as f:
    f.write(limitations)

print("Exported report tables and notes to:", EXPORT_DIR)
print(sorted(p.name for p in EXPORT_DIR.iterdir()))


In [ ]:
# ============================================================
# Regenerate all report tables from the saved CSVs on Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

ROOT     = Path("/content/drive/MyDrive/vip_cup_project")
RESULTS  = ROOT / "results"
MANIFEST = ROOT / "manifests"
EXPORTS  = ROOT / "report_exports"

comparison = pd.read_csv(RESULTS / "model_comparison.csv")
robustness = pd.read_csv(RESULTS / "robustness_results.csv")
speed      = pd.read_csv(RESULTS / "speed_benchmark.csv")
classbal   = pd.read_csv(MANIFEST / "class_balance.csv")

def show(title, df, floatfmt="%.3f"):
    print("\n" + "=" * 70 + f"\n{title}\n" + "=" * 70)
    print(df.to_string(index=False))
    print("\n--- LaTeX ---")
    print(df.to_latex(index=False, float_format=floatfmt, escape=True))

# ---- 1. Headline results (Accuracy / Precision / Recall / F1) ----
headline = comparison[comparison["model_name"].str.startswith("Model 2", na=False)][
    ["modality", "accuracy", "precision", "recall", "f1"]
].copy()
headline.columns = ["Modality", "Accuracy (mAP@0.5)", "Precision", "Recall", "F1"]
show("Headline results (per modality)", headline.sort_values("Modality"))

# ---- 2. Baseline vs neural ----
baseline = comparison[comparison["model_name"].str.contains("Baseline", na=False)][
    ["model_name", "modality", "accuracy", "precision", "recall", "f1"]
]
show("Statistical baseline (Model 1 + Gaussian Naive Bayes)", baseline)

# ---- 3. Class balance ----
pivot = classbal.pivot_table(index=["split", "modality"], columns="class_id",
                             values="count", aggfunc="sum").reset_index()
pivot.columns = ["Split", "Modality", "Class 0 (bird)", "Class 1 (drone)"]
show("Class balance", pivot, floatfmt="%.0f")
print("Totals by class:", classbal.groupby("class_id")["count"].sum().to_dict())

# ---- 4. Robustness summary (clean vs worst-case) ----
rows = []
for m in sorted(robustness["modality"].unique()):
    s = robustness[robustness["modality"] == m]
    clean = s[s["severity"] == 0]["map50"].mean()
    worst = s[s["severity"] == s["severity"].max()]["map50"].mean()
    rows.append({"Modality": m.upper(),
                 "Clean mAP@0.5": round(clean, 4),
                 "Worst-case mAP@0.5": round(worst, 4),
                 "Relative drop (%)": round(100 * (clean - worst) / clean, 1)})
show("Robustness summary", pd.DataFrame(rows), floatfmt="%.4f")

# ---- 5. Speed benchmark ----
sp = speed[["modality", "batch_size", "mean_ms_per_image", "p95_ms_per_image", "fps"]].copy()
sp.columns = ["Modality", "Batch", "Mean ms/img", "p95 ms/img", "FPS"]
show("Speed benchmark", sp.sort_values(["Modality", "Batch"]), floatfmt="%.2f")